In [1]:
# --- run me first ---
from pathlib import Path
import os
import sys

# project/notebooks -> project
if Path.cwd().name == "notebooks":
    os.chdir("..")

ROOT = Path.cwd()

# So `from src....` imports work
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("working from:", ROOT.name)

working from: project


In [2]:
import os, json, time, datetime as dt, csv, pathlib
from typing import Dict, List
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

DATA_RAW = pathlib.Path("data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

load_dotenv()

# No key yet? A free one takes about 30 seconds:
#     https://www.alphavantage.co/support/#api-key
# Put it in a .env file beside this notebook:
#     ALPHAVANTAGE_API_KEY=your_key_here
# Never hard-code it in the notebook - notebooks get shared, committed and screen-shared.
# Without a key this notebook still runs; it falls back to yfinance below.
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))

Loaded ALPHAVANTAGE_API_KEY? True


In [ ]:
# Safe timestamp for file names, e.g. 20221231-235959
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")
safe_stamp

<function __main__.safe_stamp()>

In [ ]:
# Safe filename for saving CSVs, e.g. "prefix_key1-val1_key2-val2_20221231-235959.csv"
def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

In [ ]:
# Validate a DataFrame for required columns, dtypes, and NA counts
def validate_df(df: pd.DataFrame, required_cols: List[str], dtypes_map: Dict[str, str]) -> Dict[str, str]:
    msgs = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = f"Failed to coerce {col} to {dtype}: {e}"
    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"
    return msgs

In [ ]:
# Scrape or download stock data for a given symbol, validate it, and save to CSV

SYMBOL = "AAPL"

use_alpha = bool(ALPHA_KEY)
print("Using Alpha Vantage:", use_alpha)

if use_alpha:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": SYMBOL,
        "outputsize": "compact",
        "apikey": ALPHA_KEY,
        "datatype": "json"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js.keys() if "Time Series" in k]
    if not key:
        # Alpha Vantage replies HTTP 200 with a prose blob - not an error status - when
        # the free tier's daily cap is hit or the endpoint has moved to premium, so
        # raise_for_status() above sees nothing wrong. Say so and fall back.
        print("Alpha Vantage returned no series:", str(list(js.values())[0])[:150])
        use_alpha = False

if use_alpha:
    series = js[key[0]]
    df_api = (pd.DataFrame(series).T
              .rename_axis('date')
              .reset_index())
    # keep a couple columns and coerce types
    df_api = df_api[['date', '4. close']].rename(columns={'4. close': 'close'})
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

if not use_alpha:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period="6mo", interval="1d", auto_adjust=False,
                          multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

df_api = df_api.sort_values('date').reset_index(drop=True)
msgs = validate_df(df_api, required_cols=['date','close'], dtypes_map={'date':'datetime64[ns]','close':'float'})
print(msgs)

fname = safe_filename(prefix="api", meta={"source": "alpha" if use_alpha else "yfinance", "symbol": SYMBOL})
out_path = DATA_RAW / fname
df_api.to_csv(out_path, index=False)
print("Saved:", out_path)

Using Alpha Vantage: True
{'na_total': 'Total NA values: 0'}
Saved: data/raw/api_source-alpha_symbol-AAPL_20260817-185229.csv
